# SCENTINEL — Training Random Forest untuk Klasifikasi Kesegaran Pangan

Notebook ini melatih model Random Forest dari dataset time-series yang direkam lewat
dashboard `scentinel_data_collector_v2.html`.

**Prinsip metodologis yang WAJIB diikuti di notebook ini (jangan dihapus/dilewati):**

1. **Split train/test dikelompokkan berdasarkan `batch_id`**, bukan `sample_id` atau baris acak.
   Baris dari batch fisik yang sama (mis. AYAM-A hari ke-0, ke-2, ke-4) selalu berada di sisi
   yang sama saat validasi — mencegah data leakage.
2. **Dua skema fitur dievaluasi terpisah:**
   - Skema **stabil** — rata-rata pembacaan di ujung akhir sesi (representasi performa maksimal teoretis).
   - Skema **window ≤10 detik** — hanya memakai data dari 10 detik pertama sejak sampel didekatkan,
     sesuai target kecepatan deteksi produk. Kalau performa skema ini jauh lebih buruk dari skema stabil,
     target <10 detik perlu ditinjau ulang atau fitur perlu diperkaya (mis. slope/laju perubahan).
3. **Recall kelas TIDAK_LAYAK adalah metrik utama**, bukan accuracy — false negative (busuk
   terklasifikasi LAYAK) jauh lebih berbahaya daripada false positive.
4. Karena jumlah **batch independen** kemungkinan kecil (bukan jumlah baris), gunakan **GroupKFold
   cross-validation**, bukan satu single train/test split — supaya evaluasi tidak terlalu bergantung
   pada satu pembagian yang mungkin kebetulan.

**Pengingat batasan riset (wajib tetap disebut di laporan):** SCENTINEL mendeteksi pembusukan lewat
VOC, bukan keamanan pangan. Toksin seperti cereulide (B. cereus) tidak terdeteksi sensor gas apa pun.
Label "LAYAK" berarti *tidak terdeteksi tanda pembusukan organoleptik*, bukan *aman dikonsumsi*.


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

pd.set_option('display.max_columns', None)
np.random.seed(42)


## 2. Muat Dataset

Ganti pola nama file di bawah sesuai file CSV hasil export dashboard Anda. Kalau ada beberapa
file dari beberapa sesi pengambilan data, semuanya akan digabung otomatis (asal skema kolomnya sama).


In [ ]:
# Ganti pola ini sesuai lokasi file CSV Anda.
# Contoh: 'scentinel_timeseries_*.csv' akan menggabungkan semua file yang cocok pola itu.
file_pattern = 'contoh_dataset_scentinel.csv'  # <-- GANTI ke pola file CSV Anda, mis. 'scentinel_timeseries_*.csv'

files = glob.glob(file_pattern)
if len(files) == 0:
    raise FileNotFoundError(
        f"Tidak ada file yang cocok pola '{file_pattern}'. "
        "Pastikan file CSV hasil export dashboard ada di folder yang sama dengan notebook ini, "
        "atau ubah file_pattern di atas."
    )

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print(f"Memuat {len(files)} file, total {len(df)} baris.")
df.head()


## 3. Validasi & Pembersihan Data

Cek dulu integritas dataset sebelum diproses lebih lanjut — jangan lanjut kalau ada masalah struktural.


In [ ]:
kolom_wajib = ['elapsed_ms', 'sample_id', 'batch_id', 'label',
               'mq135_ratio', 'mq3_ratio', 'mq4_ratio', 'tgs_ratio', 'suhu', 'lembap']

kolom_hilang = [k for k in kolom_wajib if k not in df.columns]
if kolom_hilang:
    raise ValueError(f"Kolom wajib hilang dari CSV: {kolom_hilang}")

print("Jumlah baris total       :", len(df))
print("Jumlah sample_id unik    :", df['sample_id'].nunique())
print("Jumlah batch_id unik     :", df['batch_id'].nunique())
print()
print("Distribusi label (per BARIS, bukan per sampel/batch):")
print(df['label'].value_counts())
print()

# Cek jumlah SAMPEL unik per label (bukan baris) — ini yang lebih bermakna
sampel_per_label = df.groupby('label')['sample_id'].nunique()
print("Jumlah SAMPEL (sample_id unik) per label:")
print(sampel_per_label)
print()

# Cek jumlah BATCH unik per label — ini yang menentukan validitas cross-validation
batch_per_label = df.groupby('label')['batch_id'].nunique()
print("Jumlah BATCH (batch_id unik) per label:")
print(batch_per_label)

if batch_per_label.min() < 3:
    print()
    print("PERINGATAN: ada kelas dengan kurang dari 3 batch independen.")
    print("Cross-validation dengan jumlah grup sekecil ini kurang stabil secara statistik —")
    print("ini keterbatasan yang WAJIB disebutkan eksplisit di laporan, bukan disembunyikan.")


## 4. Rekayasa Fitur (Feature Engineering)

Data mentah berbentuk time-series (banyak baris per sampel). Untuk klasifikasi, tiap sampel
diringkas jadi **satu baris fitur** — dilakukan dengan dua cara berbeda untuk dibandingkan.


In [ ]:
def buat_fitur_stabil(grup):
    """Skema STABIL: rata-rata pembacaan di 5 detik terakhir sesi (representasi kondisi
    mendekati steady-state, performa maksimal teoretis)."""
    t_max = grup['elapsed_ms'].max()
    window = grup[grup['elapsed_ms'] >= (t_max - 5000)]
    if len(window) == 0:
        window = grup
    return pd.Series({
        'mq135_stabil': window['mq135_ratio'].mean(),
        'mq3_stabil': window['mq3_ratio'].mean(),
        'mq4_stabil': window['mq4_ratio'].mean(),
        'tgs_stabil': window['tgs_ratio'].mean(),
        'suhu': grup['suhu'].mean(),
        'lembap': grup['lembap'].mean(),
    })


def _slope(x, y):
    """Laju perubahan (slope) linear sederhana; 0 kalau data kurang dari 2 titik."""
    if len(x) < 2 or np.ptp(x) == 0:
        return 0.0
    return np.polyfit(x, y, 1)[0]


def buat_fitur_window10s(grup):
    """Skema WINDOW 10 DETIK: hanya pakai data elapsed_ms <= 10000, sesuai target
    kecepatan deteksi produk. Menyertakan nilai terakhir dalam window DAN slope
    (laju perubahan) tiap sensor — supaya model bisa menangkap tren, bukan cuma titik akhir."""
    window = grup[grup['elapsed_ms'] <= 10000]
    if len(window) == 0:
        window = grup.iloc[[0]]
    last = window.iloc[-1]
    t = window['elapsed_ms'].values
    return pd.Series({
        'mq135_10s': last['mq135_ratio'],
        'mq3_10s': last['mq3_ratio'],
        'mq4_10s': last['mq4_ratio'],
        'tgs_10s': last['tgs_ratio'],
        'mq135_slope10s': _slope(t, window['mq135_ratio'].values),
        'mq3_slope10s': _slope(t, window['mq3_ratio'].values),
        'mq4_slope10s': _slope(t, window['mq4_ratio'].values),
        'tgs_slope10s': _slope(t, window['tgs_ratio'].values),
        'suhu': window['suhu'].mean(),
        'lembap': window['lembap'].mean(),
    })


# Kunci: satu baris per (sample_id, batch_id, label)
kunci = ['sample_id', 'batch_id', 'label']

fitur_stabil = df.groupby(kunci, dropna=False).apply(buat_fitur_stabil).reset_index()
fitur_10s = df.groupby(kunci, dropna=False).apply(buat_fitur_window10s).reset_index()

print("Fitur skema STABIL:", fitur_stabil.shape)
print("Fitur skema WINDOW 10s:", fitur_10s.shape)
fitur_stabil.head()


## 5. Fungsi Evaluasi (GroupKFold Cross-Validation)

Karena jumlah batch independen kemungkinan sedikit, dipakai **GroupKFold** — tiap fold
menjamin batch yang sama tidak pernah muncul di train dan test secara bersamaan pada fold itu.


In [ ]:
def evaluasi_skema(df_fitur, kolom_fitur, nama_skema, n_splits=4):
    X = df_fitur[kolom_fitur].values
    y = df_fitur['label'].values
    groups = df_fitur['batch_id'].values

    n_groups = df_fitur['batch_id'].nunique()
    n_splits_aktual = min(n_splits, n_groups)
    if n_splits_aktual < 2:
        print(f"[{nama_skema}] Jumlah batch terlalu sedikit untuk cross-validation (butuh minimal 2).")
        return None

    gkf = GroupKFold(n_splits=n_splits_aktual)
    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight='balanced',   # penting kalau kelas tidak seimbang
        random_state=42,
    )

    y_pred = cross_val_predict(rf, X, y, cv=gkf, groups=groups)

    print(f"===== HASIL SKEMA: {nama_skema} ({n_splits_aktual}-fold GroupKFold) =====")
    print(classification_report(y, y_pred, digits=3))

    labels_urut = sorted(y_pred.tolist() + y.tolist())
    labels_unik = sorted(set(labels_urut))
    cm = confusion_matrix(y, y_pred, labels=labels_unik)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_unik)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f"Confusion Matrix — {nama_skema}")
    plt.tight_layout()
    plt.show()

    # Fit ulang di seluruh data untuk lihat feature importance (bukan untuk klaim performa)
    rf.fit(X, y)
    importance = pd.Series(rf.feature_importances_, index=kolom_fitur).sort_values(ascending=False)
    print("Feature importance:")
    print(importance)

    return {'y_true': y, 'y_pred': y_pred, 'model': rf, 'importance': importance}


## 6. Evaluasi Skema STABIL

In [ ]:
kolom_stabil = ['mq135_stabil', 'mq3_stabil', 'mq4_stabil', 'tgs_stabil', 'suhu', 'lembap']
hasil_stabil = evaluasi_skema(fitur_stabil, kolom_stabil, "STABIL (akhir sesi)")


## 7. Evaluasi Skema WINDOW 10 DETIK

In [ ]:
kolom_10s = ['mq135_10s', 'mq3_10s', 'mq4_10s', 'tgs_10s',
             'mq135_slope10s', 'mq3_slope10s', 'mq4_slope10s', 'tgs_slope10s',
             'suhu', 'lembap']
hasil_10s = evaluasi_skema(fitur_10s, kolom_10s, "WINDOW 10 DETIK")


## 8. Bandingkan Kedua Skema

Fokus utama: **recall kelas TIDAK_LAYAK**. Kalau skema window 10 detik jauh lebih rendah recall-nya
dibanding skema stabil, itu bukti nyata target kecepatan deteksi <10 detik perlu ditinjau ulang —
jangan diabaikan sampai tahap sidang.


In [ ]:
from sklearn.metrics import recall_score

if hasil_stabil is not None and hasil_10s is not None:
    recall_stabil = recall_score(hasil_stabil['y_true'], hasil_stabil['y_pred'],
                                   labels=['TIDAK_LAYAK'], average=None, zero_division=0)[0]
    recall_10s = recall_score(hasil_10s['y_true'], hasil_10s['y_pred'],
                                labels=['TIDAK_LAYAK'], average=None, zero_division=0)[0]

    print(f"Recall kelas TIDAK_LAYAK — skema STABIL      : {recall_stabil:.3f}")
    print(f"Recall kelas TIDAK_LAYAK — skema WINDOW 10 detik: {recall_10s:.3f}")
    print()

    selisih = recall_stabil - recall_10s
    if selisih > 0.15:
        print("Selisih signifikan (>0.15) — target deteksi <10 detik kemungkinan TIDAK REALISTIS")
        print("dengan skema fitur saat ini. Pertimbangkan: (a) revisi target waktu, atau")
        print("(b) eksplorasi fitur tambahan (mis. window lebih panjang untuk slope, sensor lain).")
    else:
        print("Selisih recall kecil — target deteksi <10 detik tampak dapat dicapai dengan")
        print("skema fitur window 10 detik ini. Tetap validasi lebih lanjut dengan data lebih banyak.")


## 9. Simpan Model Final

Model final biasanya dilatih ulang di **seluruh data** (bukan cuma fold cross-validation) setelah
skema fitur dan hyperparameter final ditentukan berdasarkan hasil evaluasi di atas.


In [ ]:
import joblib

# Pilih skema yang dipakai untuk model final (ganti sesuai kesimpulan Anda dari Bagian 8)
skema_final = hasil_10s if hasil_10s is not None else hasil_stabil
model_final = skema_final['model']

joblib.dump(model_final, 'scentinel_rf_model.joblib')
print("Model tersimpan sebagai scentinel_rf_model.joblib")


## 10. Ekspor ke C untuk ESP32 (TinyML)

Random Forest **tidak didukung** TensorFlow Lite for Microcontrollers — sesuai keputusan awal
proyek, gunakan **emlearn** untuk ekspor ke kode C yang bisa langsung dimasukkan ke firmware.

Jalankan langkah ini di terminal (bukan cuma di notebook) kalau `emlearn` belum terpasang:
```bash
pip install emlearn --break-system-packages
```


In [ ]:
try:
    import emlearn

    cmodel = emlearn.convert(model_final, method='inline')
    cmodel.save(file='scentinel_rf_model.h', name='scentinel_rf')
    print("Model berhasil diekspor ke scentinel_rf_model.h")
    print("Sertakan file .h ini ke project firmware ESP32 Anda,")
    print("lalu panggil fungsi prediksi sesuai dokumentasi emlearn.")
except ImportError:
    print("Library 'emlearn' belum terpasang.")
    print("Jalankan: pip install emlearn --break-system-packages")
    print("lalu jalankan ulang sel ini.")


## Catatan Penutup

- Estimasi ukuran flash ESP32: batasi `n_estimators` dan `max_depth` (sudah diset 100 dan 6 di atas)
  supaya model hasil ekspor emlearn tidak membengkak — kalau ukuran `.h` terlalu besar, turunkan
  `n_estimators` dan uji ulang trade-off akurasi vs ukuran.
- **Jangan laporkan angka accuracy tunggal tanpa disertai recall per kelas** — terutama recall
  TIDAK_LAYAK, sesuai prinsip yang sudah disepakati sejak awal proyek ini.
- Kalau jumlah batch independen per kelas masih sangat sedikit (<3-5), cantumkan ini sebagai
  **keterbatasan riset eksplisit** di laporan — bukan diam-diam disembunyikan di balik jumlah baris
  yang kelihatan banyak.
- Ingat batasan mendasar: SCENTINEL mendeteksi pembusukan VOC, bukan keamanan pangan (cereulide
  dan toksin non-volatil lain tidak terdeteksi sensor gas apa pun).
